# InduSense de bout en bout

**Maintenance prédictive : des relevés capteurs à la décision de réentraîner, en un seul notebook.**

Ce notebook raconte tout le projet **sans réécrire de logique** : chaque cellule appelle le
package `indusense` (dossier `src/indusense/`), celui-là même qui tourne dans l'API, dans les
flows Prefect et dans l'image Docker. Les notebooks TP1 à TP12 restent l'historique de
construction ; celui-ci est la **vitrine**.

| Étape | Domaine | Modules du package | Question métier |
|---|---|---|---|
| 1 | Données | `ingest`, `processing/*`, `flows/etl_flow`, `data` | Les relevés sont-ils propres et prêts pour le modèle ? |
| 2 | Modèle | `modeling/*` | Le modèle détecte-t-il les pannes ? |
| 3 | Prédiction | `scoring`, `predictions_store`, `api/main` | Quelle machine risque une panne dans les 24 h ? |
| 4 | Retour humain | `predictions_store.review_prediction` | Le technicien confirme-t-il l'alerte ? |
| 5 | Cycle de vie | `drift`, `flows/retrain_flow`, `arbitration`, `shadow` | Faut-il réentraîner, et remplacer le modèle ? |

Sous toutes les étapes, un socle : `config.py` (où est la base, où est le modèle…).

### Conventions de lecture

- 🟢 **Lecture seule** : la cellule lit la base ou les fichiers, elle ne modifie rien.
- 🧪 **Bac à sable** : la cellule écrit, mais dans une base **temporaire** jetée à la fin.
- 🟠 **Écrit pour de vrai** : désactivée par défaut (un drapeau `RUN_... = False` à passer à `True`).

On peut donc exécuter tout le notebook sans risque : **rien n'est modifié** dans la base du
projet, ni dans `model.joblib`.

### Prérequis

1. Docker Desktop lancé, base démarrée : `docker compose start` dans le dossier `Docker/`
   (le service Windows `postgresql-x64-18` doit être arrêté : il occupe le port 5432).
2. Noyau Jupyter : le Python du projet (`ML/.venv`).
3. Données DVC présentes : `uv run --project ML dvc pull` si besoin.

## 0. Préparer l'environnement

On importe le socle `config.py` et on vérifie que la base répond. Si cette cellule échoue,
inutile d'aller plus loin : revoir les prérequis ci-dessus.

In [1]:
# 🟢 Lecture seule
import warnings
from pathlib import Path

import pandas as pd
from sqlalchemy import text

from indusense.config import (
    ML_DIR,
    get_engine,
    get_model_path,
    get_model_version,
    get_predictions_db_url,
)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)  # avertissements techniques pandas / sklearn
pd.set_option("display.max_columns", 12)

engine = get_engine()
with engine.connect() as conn:
    version = conn.execute(text("SELECT version()")).scalar()

print("Dossier du projet :", ML_DIR)
print("Base PostgreSQL   :", version.split(",")[0])
print("Modèle servi      :", get_model_path().relative_to(ML_DIR))
print("Base prédictions  :", get_predictions_db_url().split("///")[-1].replace(str(ML_DIR), "ML"))

Dossier du projet : C:\Users\Moi\Desktop\TRAINING\py-init\ML
Base PostgreSQL   : PostgreSQL 15.1 (Debian 15.1-1.pgdg110+1) on x86_64-pc-linux-gnu
Modèle servi      : artifacts\models\model.joblib
Base prédictions  : ML\artifacts\predictions.db


> **À retenir** : aucune adresse ni mot de passe n'est écrit dans ce notebook. Tout vient de
> `config.py`, qui lit les variables d'environnement (fichier `.env`) avec un défaut sûr.

---
# Étape 1 : Données (Bronze → Silver → Gold)

Les données suivent une architecture en **médaillon**, à trois étages :

| Étage | Contenu | Règle |
|---|---|---|
| **Bronze** | les fichiers terrain, tels que reçus (texte brut) | on ne corrige rien, on garde tout |
| **Silver** | des données typées, nettoyées, avec des **drapeaux qualité** | on signale, on ne supprime pas en silence |
| **Gold** | une ligne par **machine et par heure**, avec ~80 indicateurs | prêt pour le modèle |

Chaque passage d'un étage à l'autre est inscrit dans un **registre**, la table
`ingestion_batch` : qui a produit quoi, quand, combien de lignes.

## 1.1 Bronze : ce qui est arrivé du terrain

Trois sources : la télémétrie des capteurs, les relevés d'incidents, et les maintenances.

In [2]:
# 🟢 Lecture seule
with engine.connect() as conn:
    bronze = {
        t: conn.execute(text(f"SELECT count(*) FROM {t}")).scalar()
        for t in ["bronze_telemetry", "bronze_incidents", "bronze_maintenance"]
    }
pd.Series(bronze, name="lignes").to_frame()

,lignes
bronze_telemetry,135626
bronze_incidents,1245
bronze_maintenance,1562


Un nouveau lot terrain entre en Bronze par `indusense.ingest.ingest_terrain_batch(engine, csv, source)`.
Deux propriétés importantes :

- **Ajout (append)** : il ne réécrit pas le Bronze existant, il ajoute le lot.
- **Idempotent** : il calcule l'empreinte (MD5) du fichier. Si ce fichier exact a déjà été
  ingéré, il ne fait rien. Envoyer deux fois le même fichier ne crée donc aucun doublon.

On ne l'exécute pas ici (cela ajouterait des lignes au Bronze du projet). Voici le registre
des lots déjà passés :

In [3]:
# 🟢 Lecture seule
registre = pd.read_sql(
    "SELECT source_name, status, started_at, rows_loaded FROM ingestion_batch ORDER BY started_at",
    engine,
)
registre.tail(10)

,source_name,status,started_at,rows_loaded
5,silver_maintenance,done,2026-09-23 12:14:39.724309+00:00,1562
6,gold,done,2026-09-23 12:19:14.429325+00:00,132940
7,silver_telemetry,done,2026-09-24 09:42:56.376153+00:00,661920
8,silver_incidents,done,2026-09-24 09:45:07.732545+00:00,1245
9,silver_maintenance,done,2026-09-24 09:45:08.010197+00:00,1562
10,gold,done,2026-09-24 09:49:01.138723+00:00,132940
11,silver_telemetry,done,2026-09-24 09:53:24.378000+00:00,661920
12,silver_incidents,done,2026-09-24 09:55:24.766057+00:00,1245
13,silver_maintenance,done,2026-09-24 09:55:25.043279+00:00,1562
14,gold,done,2026-09-24 09:59:17.240262+00:00,132940


> **Lire le registre** : les lignes `telemetry`, `incidents`, `maintenance` sont des arrivées
> en Bronze ; les lignes `silver_*` et `gold` sont des reconstructions par le flow ETL.
> La ligne `gold` est importante : c'est elle que regarde le **feu vert n°5** du réentraînement
> (étape 5).

## 1.2 Silver capteurs : du format large au format long

Le Bronze télémétrie a **une ligne par relevé** et une colonne par capteur. Le Silver passe en
**format long** : une ligne par relevé **et par capteur**, avec trois drapeaux qualité :

- `is_missing` : valeur absente ;
- `is_duplicate` : même machine, même instant, même capteur, déjà vu ;
- `is_outlier` : hors de l'intervalle `[Q1 − 1,5 × IQR ; Q3 + 1,5 × IQR]` de son capteur.

La fonction `build_silver_sensor_readings` est **pure** : elle prend un tableau et rend un
tableau, sans lire ni écrire en base. On peut donc l'appeler ici, en mémoire, sans risque.

In [4]:
# 🟢 Lecture seule (calcul en mémoire)
from indusense.flows.etl_flow import BRONZE_TELEMETRY_QUERY
from indusense.processing.silver_sensor_reading import build_silver_sensor_readings

bronze_tel = pd.read_sql(text(BRONZE_TELEMETRY_QUERY), engine)
silver_sensors = build_silver_sensor_readings(bronze_tel)

print(f"Bronze télémétrie : {len(bronze_tel):>7} lignes (format large)")
print(f"Silver capteurs   : {len(silver_sensors):>7} lignes (format long)")
silver_sensors.head()

Bronze télémétrie :  132940 lignes (format large)
Silver capteurs   :  664700 lignes (format long)


,machine_id,observed_at,sensor_type,sensor_value,unit,is_missing,is_duplicate,is_outlier
0,MACH-01,2025-06-01 01:00:00+00:00,temperature_c,48.762,°C,False,False,False
1,MACH-01,2025-06-01 02:00:00+00:00,temperature_c,51.352,°C,False,False,False
2,MACH-01,2025-06-01 03:00:00+00:00,temperature_c,49.512,°C,False,False,False
3,MACH-01,2025-06-01 04:00:00+00:00,temperature_c,51.982,°C,False,False,False
4,MACH-01,2025-06-01 05:00:00+00:00,temperature_c,50.402,°C,False,False,False


In [5]:
# 🟢 Lecture seule : combien de lignes chaque drapeau signale-t-il ?
flags = silver_sensors[["is_missing", "is_duplicate", "is_outlier"]].sum().rename("lignes signalées")
exclues = int((silver_sensors["is_missing"] | silver_sensors["is_duplicate"]).sum())
print(f"Écartées du Gold (manquantes ou doublons) : {exclues}")
print(f"Gardées pour le Gold                      : {len(silver_sensors) - exclues}")
flags.to_frame()

Écartées du Gold (manquantes ou doublons) : 2780
Gardées pour le Gold                      : 661920


,lignes signalées
is_missing,2780
is_duplicate,0
is_outlier,3237


> **Pourquoi signaler plutôt que supprimer ?** Une valeur aberrante peut être une vraie panne
> qui commence. On la garde, on la marque, et chaque étage suivant décide quoi en faire.
> Seules les lignes manquantes ou en double sont écartées du Gold.

## 1.3 Silver événements : incidents et maintenances

Même principe pour les deux autres sources (`processing/silver_events.py`) :

- les **incidents** sont horodatés en UTC, rattachés à leur opérateur (par un nom anonymisé)
  et dédoublonnés ; ceux de gravité ≥ 4 deviennent des **événements de label** : ce sont eux
  que le modèle apprendra à anticiper ;
- les **maintenances** sont datées en UTC et dédoublonnées.

In [6]:
# 🟢 Lecture seule (calcul en mémoire)
from indusense.flows.etl_flow import (
    BRONZE_INCIDENTS_QUERY,
    BRONZE_MAINTENANCE_QUERY,
    OPERATOR_QUERY,
)
from indusense.processing.silver_events import build_silver_incidents, build_silver_maintenance

silver_inc = build_silver_incidents(
    pd.read_sql(text(BRONZE_INCIDENTS_QUERY), engine),
    pd.read_sql(text(OPERATOR_QUERY), engine),
)
silver_maint = build_silver_maintenance(pd.read_sql(text(BRONZE_MAINTENANCE_QUERY), engine))

print(f"Incidents    : {len(silver_inc)} dont {int(silver_inc['is_label_event'].sum())} événements de label")
print(f"Maintenances : {len(silver_maint)}")
silver_inc.head()

Incidents    : 1245 dont 205 événements de label
Maintenances : 1562


,incident_code,machine_id,operator_id,occurred_at,severity,shift,comment,is_label_event
0,INC-000001,MACH-06,1,2025-06-01 05:42:00+00:00,4,nuit,chauffe anormale,True
1,INC-000002,MACH-15,2,2025-06-01 21:08:00+00:00,3,apres-midi,micro-fuite / baisse pression,False
2,INC-000003,MACH-10,1,2025-06-02 05:43:00+00:00,2,nuit,défaut capteur confirmé,False
3,INC-000004,MACH-10,3,2025-06-03 05:43:00+00:00,3,nuit,signal capteur hors plage,False
4,INC-000005,MACH-14,4,2025-06-04 01:01:00+00:00,2,nuit,alerte température haute,False


## 1.4 Gold : une ligne par machine et par heure

`processing/gold_features.py` transforme les trois Silver en **indicateurs horaires** :

| Famille | Exemples | Idée |
|---|---|---|
| Fenêtres glissantes | `temp_mean_24h`, `pressure_std_6h` | le comportement récent |
| Tendances | `temp_delta_1h`, `rotation_trend_6h` | ça monte, ça descend ? |
| Z-scores | `temp_zscore_machine` | anormal **pour cette machine** ? |
| Historique | `incident_count_prev_7d`, `days_since_last_maintenance` | le passé de la machine |
| Labels | `label_failure_next_24h` | **la cible** : une panne dans les 24 h qui suivent ? |

On recalcule le Gold en mémoire, à partir des tables Silver, avec les **mêmes requêtes** que le
flow ETL (constantes importées de `etl_flow`). Compter une trentaine de secondes.

In [7]:
# 🟢 Lecture seule (calcul en mémoire, ~30 s)
from indusense.flows.etl_flow import (
    MACHINE_QUERY,
    SILVER_INCIDENTS_QUERY,
    SILVER_MAINTENANCE_QUERY,
    SILVER_SENSORS_QUERY,
)
from indusense.processing.gold_features import build_gold_features, to_gold_rows

gold_memoire = to_gold_rows(
    build_gold_features(
        pd.read_sql(text(SILVER_SENSORS_QUERY), engine, parse_dates=["observed_at"]),
        pd.read_sql(text(SILVER_INCIDENTS_QUERY), engine, parse_dates=["occurred_at"]),
        pd.read_sql(text(SILVER_MAINTENANCE_QUERY), engine, parse_dates=["performed_at"]),
        pd.read_sql(text(MACHINE_QUERY), engine),
    )
)
with engine.connect() as conn:
    n_table = conn.execute(text("SELECT count(*) FROM gold_machine_hourly_feature")).scalar()

print(f"Gold recalculé en mémoire : {gold_memoire.shape[0]} lignes × {gold_memoire.shape[1]} colonnes")
print(f"Table Gold en base        : {n_table} lignes")
print(f"Machines                  : {gold_memoire['machine_id'].nunique()}")
print(f"Taux de pannes à 24 h     : {gold_memoire['label_failure_next_24h'].mean():.1%}")

Gold recalculé en mémoire : 132940 lignes × 90 colonnes
Table Gold en base        : 132940 lignes
Machines                  : 15
Taux de pannes à 24 h     : 3.6%


> **Un jeu très déséquilibré** : moins de 4 % des heures précèdent une panne. Un modèle qui
> dirait toujours « pas de panne » aurait plus de 96 % de bonnes réponses… et ne servirait à
> rien. C'est pourquoi on juge le modèle sur le **rappel** et la **PR-AUC**, pas sur
> l'exactitude (étape 2).

## 1.5 Le flow ETL : tout l'étage Données en une commande

Jusqu'ici, on a appelé les fonctions une par une, en mémoire. En production, c'est le flow
Prefect `indusense-etl` (`flows/etl_flow.py`) qui les enchaîne **et écrit en base** :

```
silver-sensor-reading ─┐
silver-incident ───────┼──▶ gold
silver-maintenance ────┘
```

Chaque étage :
1. ouvre un lot dans `ingestion_batch` ;
2. vide et réécrit sa table **dans une seule transaction** (si ça échoue, l'ancienne version
   reste en place, jamais une table vide) ;
3. ferme le lot avec ses compteurs.

Commande équivalente dans un terminal : `uv run --frozen indusense etl`.

C'est la correction de la **rupture n°1** : avant, il fallait relancer les notebooks TP4 à TP6.

🟠 La cellule suivante **réécrit** les tables Silver et Gold (environ 7 minutes). Passer
`RUN_ETL = True` pour l'exécuter.

In [8]:
# 🟠 Écrit pour de vrai (désactivé par défaut)
RUN_ETL = False

if RUN_ETL:
    from indusense.flows.etl_flow import etl_flow

    print(etl_flow())
else:
    print("Flow ETL non lancé (RUN_ETL = False).")

Flow ETL non lancé (RUN_ETL = False).


## 1.6 Figer un instantané : le Gold exporté et son empreinte

Pour entraîner et pour tracer, on **fige** le Gold dans un fichier CSV
(`data.export_gold_dataset`), versionné par DVC. Son empreinte MD5 (`data.hash_file`) est
notée dans l'étiquette `data/gold/gold_dataset.csv.dvc`.

**Question** : le Gold reconstruit par le flow ETL est-il le même que celui versionné par DVC,
sur lequel le modèle servi a été entraîné ? On exporte le Gold actuel dans un dossier temporaire
(pour ne pas toucher au fichier suivi par DVC), puis on compare de deux façons : l'empreinte du
fichier, et le contenu, colonne par colonne.

In [9]:
# 🧪 Bac à sable : export dans un dossier temporaire (~144 Mo, supprimé ensuite)
import tempfile

import yaml

from indusense.data import export_gold_dataset, hash_file

with tempfile.TemporaryDirectory() as tmp:
    export = export_gold_dataset(engine, Path(tmp) / "gold.csv")
    md5_frais = hash_file(export)
    gold_frais = pd.read_csv(export)
gold_dvc = pd.read_csv(ML_DIR / "data/gold/gold_dataset.csv")
md5_dvc = yaml.safe_load((ML_DIR / "data/gold/gold_dataset.csv.dvc").read_text())["outs"][0]["md5"]

colonnes_differentes = [
    c for c in gold_dvc.columns if not gold_dvc[c].astype(str).equals(gold_frais[c].astype(str))
]
print("Empreinte du Gold exporté maintenant :", md5_frais)
print("Empreinte notée dans l'étiquette DVC :", md5_dvc)
print("Empreintes identiques                :", md5_frais == md5_dvc)
print()
print(f"Même taille                          : {gold_dvc.shape == gold_frais.shape} {gold_frais.shape}")
print("Colonnes dont le contenu diffère     :", colonnes_differentes)

Empreinte du Gold exporté maintenant : faea72a52673c648d790aa12c7cccaa7
Empreinte notée dans l'étiquette DVC : 47f3185b0314adc00960476486f177fa
Empreintes identiques                : False

Même taille                          : True (132940, 92)
Colonnes dont le contenu diffère     : ['ingestion_batch_id']


### Lecture du résultat

Les empreintes **diffèrent**, et pourtant **une seule colonne** change : `ingestion_batch_id`,
l'identifiant du lot qui a écrit la table. Chaque exécution du flow ETL crée un nouveau lot,
donc un nouvel identifiant sur les 132 940 lignes, donc un fichier différent octet par octet.

**Les données métier (les ~90 indicateurs et les labels) sont identiques.** La rupture n°1 est
bien corrigée sans rien changer au résultat.

> **Piste d'amélioration** : tel quel, DVC verrait une « nouvelle version » du Gold après chaque
> ETL, même quand rien n'a changé. Exclure la colonne technique `ingestion_batch_id` de l'export
> rendrait l'empreinte stable : même contenu ⇒ même empreinte.

---
# Étape 2 : Modèle

## 2.1 Préparer les données d'entraînement

`modeling/dataset.py` charge le Gold et le découpe :

- **features** (`X`) : les indicateurs, **sans** les colonnes qui trahiraient la réponse
  (labels des autres horizons, compteurs d'incidents futurs) : c'est la lutte contre la **fuite
  de données** ;
- **cible** (`y`) : `label_failure_next_24h` ;
- **découpage temporel** : `train` + `validation` pour apprendre, `test` pour juger, sur une
  période **postérieure**. On ne teste jamais sur le passé de ce qu'on a appris.

In [10]:
# 🟢 Lecture seule
from indusense.modeling.dataset import TARGET, load_gold_dataset

gold = load_gold_dataset(engine)

pd.DataFrame(
    {
        "lignes": [len(gold.X_tv), len(gold.X_test)],
        "pannes": [int(gold.y_tv.sum()), int(gold.y_test.sum())],
        "début": [gold.trainval_df["window_start"].min(), gold.test_df["window_start"].min()],
        "fin": [gold.trainval_df["window_start"].max(), gold.test_df["window_start"].max()],
    },
    index=["apprentissage (train + validation)", "test"],
).assign(features=len(gold.feature_cols))

,lignes,pannes,début,fin,features
apprentissage (train + validation),112996,4018,2025-06-01 00:00:00+00:00,2026-04-14 00:00:00+00:00,78
test,19944,727,2026-04-14 01:00:00+00:00,2026-06-08 23:00:00+00:00,78


## 2.2 Le modèle : imputation + XGBoost

`modeling/pipeline.py` construit un pipeline scikit-learn en deux temps :

1. **Imputation par la médiane** : un capteur muet ne fait pas planter le modèle ;
2. **XGBoost** : des centaines de petits arbres de décision qui se corrigent les uns les autres.

Les réglages retenus (`B11_PARAMS`) viennent de l'optimisation Optuna des TP. Un réglage clé :
`scale_pos_weight`, qui donne plus de poids aux pannes, rares, pour que le modèle ne les ignore
pas.

In [11]:
# 🟢 Lecture seule
from indusense.modeling.train import B11_PARAMS, compute_scale_pos_weight

params = {**B11_PARAMS, "scale_pos_weight": compute_scale_pos_weight(gold.y_tv)}
pd.Series(params, name="valeur").to_frame()

,valeur
n_estimators,287.000000
max_depth,9.000000
learning_rate,0.028871
subsample,0.825202
colsample_bytree,0.573631
min_child_weight,11.000000
reg_alpha,0.019185
reg_lambda,0.622876
random_state,42.000000
verbosity,0.000000


## 2.3 Entraîner et évaluer

`train_and_evaluate` entraîne sur la période d'apprentissage et mesure sur la période de test.
Le modèle entraîné reste **en mémoire** : on ne remplace pas `model.joblib`.

On compare trois choses, sur la même période de test :

1. les **métriques de référence** enregistrées avec le modèle servi (`artifacts/models/metrics.json`) ;
2. le **modèle servi** (`model.joblib`) évalué maintenant sur le Gold actuel ;
3. un **modèle réentraîné** maintenant, sur ce PC, avec les mêmes données et les mêmes réglages.

In [12]:
# 🟢 Lecture seule (entraînement en mémoire, ~1 min)
import json

import joblib
from sklearn.metrics import average_precision_score, confusion_matrix, precision_score, recall_score

from indusense.modeling.train import train_and_evaluate

reference = json.loads((ML_DIR / "artifacts/models/metrics.json").read_text())

servi = joblib.load(get_model_path())
pred_servi = servi.predict(gold.X_test)
tn, fp, fn, tp = confusion_matrix(gold.y_test, pred_servi).ravel()
metriques_servi = {
    "recall_test": round(recall_score(gold.y_test, pred_servi), 4),
    "tp": tp, "fn": fn, "fp": fp,
    "precision_test": round(precision_score(gold.y_test, pred_servi), 4),
    "pr_auc_test": round(average_precision_score(gold.y_test, servi.predict_proba(gold.X_test)[:, 1]), 4),
}

pipe, metrics = train_and_evaluate(gold.X_tv, gold.y_tv, gold.X_test, gold.y_test, params)

cles = ["recall_test", "tp", "fn", "fp", "precision_test", "pr_auc_test"]
pd.DataFrame(
    {
        "1. référence (metrics.json)": {k: reference[k] for k in cles},
        "2. modèle servi, évalué maintenant": metriques_servi,
        "3. réentraîné maintenant sur ce PC": {k: metrics[k] for k in cles},
    }
).loc[cles]

,1. référence (metrics.json),"2. modèle servi, évalué maintenant",3. réentraîné maintenant sur ce PC
recall_test,0.9106,0.9106,0.9161
tp,662.0000,662.0000,666.0000
fn,65.0000,65.0000,61.0000
fp,255.0000,255.0000,283.0000
precision_test,0.7219,0.7219,0.7018
pr_auc_test,0.8945,0.8945,0.8997


### Lecture du résultat

- **Colonnes 1 et 2 identiques** : le modèle servi, évalué sur le Gold reconstruit par le flow,
  retrouve exactement ses métriques de référence. C'est une preuve de plus que le Gold est le
  même.
- **Colonne 3 légèrement différente** : quelques pannes détectées ou fausses alertes d'écart,
  alors que les données et les réglages sont identiques et la graine fixée
  (`random_state=42`). L'entraînement de XGBoost dépend aussi de l'**environnement** :
  processeur, nombre de cœurs, ordre des calculs en parallèle. Le modèle servi a été entraîné
  sur une autre machine.

> **Leçon** : « mêmes données + mêmes réglages » ne garantit pas un modèle identique au bit
> près. C'est précisément pour ça qu'on **versionne le fichier du modèle** (`model.joblib`,
> suivi par DVC) au lieu de compter sur un réentraînement pour le retrouver.

### Lire ces métriques

| Métrique | Question | Pourquoi elle compte ici |
|---|---|---|
| **Rappel** (`recall_test`) | Sur 100 vraies pannes, combien détectées ? | Une panne manquée coûte cher : c'est l'indicateur n°1 |
| **Précision** | Sur 100 alertes, combien justes ? | Trop de fausses alertes et les techniciens n'y croient plus |
| **PR-AUC** | Qualité globale du classement des risques | Adaptée aux classes rares |
| `tp` / `fn` / `fp` | Pannes détectées / manquées / fausses alertes | Les mêmes chiffres qu'on retrouve à l'étape 4 |

`modeling/tracking.py` enregistrerait ce run dans **MLflow**, avec l'empreinte du Gold. On ne
le fait pas ici pour ne pas ajouter un run d'exercice à l'historique.

---
# Étape 3 : Prédiction

Deux façons de prédire avec le **même** modèle :

- **le facteur (en lot)** : le flow `indusense-pipeline` passe chez toutes les machines à heure
  fixe et dépose les prédictions dans la table `predictions` ;
- **le guichet (en ligne)** : l'API répond à une question précise, tout de suite.

On rejoue ici les étapes du facteur, puis on vérifie que le guichet donne le même chiffre.

## 3.1 Charger le modèle servi et sa version

La **version** est l'empreinte du fichier `model.joblib` (12 caractères). Elle accompagne
chaque prédiction : on saura toujours quel modèle a produit quel chiffre.

In [13]:
# 🟢 Lecture seule
import joblib

model = joblib.load(get_model_path())
model_version = get_model_version()
print("Version du modèle servi :", model_version)

Version du modèle servi : f550cff24814


## 3.2 La dernière heure connue de chaque machine

C'est l'étape `load_latest_features` du flow : parmi les ~133 000 lignes du Gold, on garde la
plus récente de chaque machine.

In [14]:
# 🟢 Lecture seule
latest = gold_memoire.sort_values("window_start").groupby("machine_id", as_index=False).tail(1)
print(f"{len(gold_memoire)} lignes → {len(latest)} (une par machine)")
latest[["machine_id", "window_start", "temp_mean_24h", "pressure_mean_24h"]].head()

132940 lignes → 15 (une par machine)


,machine_id,window_start,temp_mean_24h,pressure_mean_24h
79749,MACH-09,2026-06-08 23:00:00+00:00,47.906417,200.411250
53172,MACH-06,2026-06-08 23:00:00+00:00,46.217083,200.155375
88611,MACH-10,2026-06-08 23:00:00+00:00,47.473833,200.044667
124071,MACH-14,2026-06-08 23:00:00+00:00,47.180917,200.013500
17734,MACH-02,2026-06-08 23:00:00+00:00,44.683833,200.251250


## 3.3 Calculer les probabilités : `scoring.score_features`

`score_features` fait deux choses :

1. il présente au modèle **exactement** les colonnes apprises, dans le bon ordre
   (`model_feature_cols`) ;
2. il renvoie la probabilité de panne **et** une photo JSON des features utilisées
   (`features_payload`), pour pouvoir rejouer cette prédiction plus tard.

In [15]:
# 🟢 Lecture seule
from indusense.scoring import score_features

proba, payloads = score_features(model, latest)

predictions = latest[["machine_id", "window_start"]].assign(
    failure_proba_24h=proba.round(4), model_version=model_version
)
print("Machines en alerte (probabilité ≥ 0,5) :", int((proba >= 0.5).sum()), "sur", len(proba))
predictions.sort_values("failure_proba_24h", ascending=False).head(10)

Machines en alerte (probabilité ≥ 0,5) : 0 sur 15


,machine_id,window_start,failure_proba_24h,model_version
115211,MACH-13,2026-06-08 23:00:00+00:00,0.0142,f550cff24814
53172,MACH-06,2026-06-08 23:00:00+00:00,0.0004,f550cff24814
62042,MACH-07,2026-06-08 23:00:00+00:00,0.0004,f550cff24814
132939,MACH-15,2026-06-08 23:00:00+00:00,0.0004,f550cff24814
26584,MACH-03,2026-06-08 23:00:00+00:00,0.0003,f550cff24814
88611,MACH-10,2026-06-08 23:00:00+00:00,0.0003,f550cff24814
79749,MACH-09,2026-06-08 23:00:00+00:00,0.0002,f550cff24814
8870,MACH-01,2026-06-08 23:00:00+00:00,0.0002,f550cff24814
17734,MACH-02,2026-06-08 23:00:00+00:00,0.0002,f550cff24814
124071,MACH-14,2026-06-08 23:00:00+00:00,0.0002,f550cff24814


> **Seuil de décision** : au-delà de 0,5, la machine est en alerte. Ce seuil est **gelé**
> dans `metrics.json` : on ne le modifie jamais pour faire taire une alerte (voir le runbook).
> À la dernière heure du jeu de données, le parc est calme : les probabilités restent faibles.

## 3.4 Déposer dans la boîte aux lettres, sans doublon

`predictions_store.upsert_predictions` range chaque prédiction sous la clé
`(machine_id, window_start)` :

- **nouvelle clé** → nouvelle ligne ;
- **clé existante** → la ligne est **mise à jour**, jamais dupliquée ;
- les colonnes de **revue technicien** ne sont **jamais** écrasées.

Démonstration dans une base SQLite **temporaire** : on dépose deux fois les mêmes prédictions.

In [16]:
# 🧪 Bac à sable : base SQLite temporaire
from datetime import UTC, datetime

from sqlalchemy import create_engine

from indusense.predictions_store import upsert_predictions

bac_a_sable = Path(tempfile.mkdtemp()) / "predictions_demo.db"
demo_engine = create_engine(f"sqlite:///{bac_a_sable}")

rows = [
    {
        "machine_id": r.machine_id,
        "window_start": r.window_start.isoformat(),
        "failure_proba_24h": float(p),
        "scored_at": datetime.now(UTC).isoformat(),
        "model_version": model_version,
        "features_payload": payload,
    }
    for r, p, payload in zip(latest.itertuples(), proba, payloads)
]

print("Après le 1er dépôt :", upsert_predictions(demo_engine, rows), "lignes")
print("Après le 2e dépôt  :", upsert_predictions(demo_engine, rows), "lignes (aucun doublon)")

Après le 1er dépôt : 15 lignes
Après le 2e dépôt  : 15 lignes (aucun doublon)


## 3.5 Le guichet : la même réponse par l'API

On interroge l'API FastAPI **sans lancer de serveur**, grâce au client de test de FastAPI.
On envoie les features de la machine la plus à risque, avec la clé d'API.

Depuis le commit `b9e271e`, l'API choisit ses colonnes avec la même fonction que le facteur
(`scoring.model_feature_cols`) : les deux chiffres doivent être **identiques**.

In [17]:
# 🟢 Lecture seule (aucun serveur lancé, rien n'est stocké)
from fastapi.testclient import TestClient

from indusense.api.main import app
from indusense.config import get_api_key

i = int(proba.argmax())
machine = latest.iloc[i]["machine_id"]

client = TestClient(app)
reponse = client.post(
    "/predict-tabular", json={"features": payloads[i]}, headers={"X-API-Key": get_api_key()}
)
sans_cle = client.post("/predict-tabular", json={"features": payloads[i]})

print("Machine                :", machine)
print("Le facteur (lot)       :", round(float(proba[i]), 6))
print("Le guichet (API)       :", round(reponse.json()["failure_proba_24h"], 6))
print("Sans clé d'API         : code HTTP", sans_cle.status_code)

Machine                : MACH-13
Le facteur (lot)       : 0.014206
Le guichet (API)       : 0.014206
Sans clé d'API         : code HTTP 401


> **Limite du guichet (rupture n°3)** : la réponse de l'API n'est **pas** enregistrée dans la
> table `predictions`. Elle ne peut donc pas être revue par un technicien (étape 4) ni compter
> pour le réentraînement (étape 5). Voir `facteur_guichet_pedagogique.pptx` pour le dossier de
> décision.

---
# Étape 4 : Retour humain (Human In The Loop)

Une prédiction n'est qu'une hypothèse. Le **technicien** tranche, dans l'interface Streamlit
(`scripts/streamlit_review.py`), qui appelle `predictions_store.review_prediction`. Chaque
prédiction reçoit un statut :

| Statut | Signification |
|---|---|
| `A_VALIDER` | pas encore revue |
| `PANNE_CONFIRMEE` | alerte juste : la panne a bien eu lieu |
| `FAUSSE_ALERTE` | alerte, mais pas de panne |
| `INCIDENT_NON_PREDIT` | panne que le modèle n'a pas vue : **le retour le plus précieux** |

In [18]:
# 🟢 Lecture seule : l'état réel de la base des prédictions
from indusense.config import get_predictions_engine

statuts = pd.read_sql(
    "SELECT review_status, count(*) AS n FROM predictions GROUP BY review_status ORDER BY n DESC",
    get_predictions_engine(),
)
statuts

,review_status,n
0,A_VALIDER,18962
1,PANNE_CONFIRMEE,662
2,FAUSSE_ALERTE,255
3,INCIDENT_NON_PREDIT,65


> Comparez avec le modèle servi à l'étape 2.3 : 662 pannes confirmées = `tp`, 255 fausses alertes = `fp`,
> 65 incidents non prédits = `fn`. Les verdicts des techniciens et les métriques du modèle
> racontent la même histoire.

Démonstration de `review_prediction` dans le bac à sable : sur la machine la plus à risque,
que le modèle n'avait pourtant pas mise en alerte, un technicien constate une panne. Il la
déclare comme **incident non prédit**. Puis le facteur repasse et redépose ses prédictions :
**le verdict n'est pas effacé**.

In [19]:
# 🧪 Bac à sable
from indusense.predictions_store import review_prediction

review_prediction(
    demo_engine,
    machine_id=rows[i]["machine_id"],
    window_start=rows[i]["window_start"],
    review_status="INCIDENT_NON_PREDIT",
    ground_truth=True,
    reviewer_comment="Fuite hydraulique constatée à la ronde",
    reviewed_at=datetime.now(UTC).isoformat(),
)
upsert_predictions(demo_engine, rows)  # le facteur repasse

pd.read_sql(
    "SELECT machine_id, failure_proba_24h, review_status, reviewer_comment FROM predictions "
    "WHERE review_status != 'A_VALIDER'",
    demo_engine,
)

,machine_id,failure_proba_24h,review_status,reviewer_comment
0,MACH-13,0.014206,INCIDENT_NON_PREDIT,Fuite hydraulique constatée à la ronde


---
# Étape 5 : Cycle de vie du modèle

Un modèle vieillit : les machines s'usent, les conditions changent. Le flow
`indusense-retrain-cycle` (`flows/retrain_flow.py`) décide **quand** réentraîner, et
`arbitration.py` décide **si** le nouveau modèle mérite de remplacer l'ancien.

```
5 feux verts ──▶ arbitrage champion / challenger ──▶ mode fantôme ──▶ bascule
```

## 5.1 Mesurer la dérive : le PSI

Le **PSI** (*Population Stability Index*, `drift.psi`) compare la distribution d'une feature
entre une période de **référence** (l'apprentissage) et une période **courante**.

| PSI | Lecture |
|---|---|
| < 0,1 | stable |
| 0,1 à 0,25 | à surveiller |
| > 0,25 | **dérive** : seuil d'alerte du projet |

In [20]:
# 🟢 Lecture seule
from indusense.arbitration import CUTOFF
from indusense.drift import drift_table
from indusense.flows.retrain_flow import DRIFT_FEATURES

courant = gold.test_df[gold.test_df["window_start"] < CUTOFF]
table_derive = drift_table(gold.trainval_df, courant, DRIFT_FEATURES)
table_derive.assign(alerte=table_derive["psi"] > 0.25).round(4)

,feature,psi,ks_pvalue,alerte
0,temp_mean_24h,0.3841,0.0000,True
1,temp_std_24h,0.0046,0.0006,False
2,pressure_mean_24h,0.0080,0.0001,False
3,pressure_std_24h,0.0073,0.0000,False
4,voltage_mean_24h,0.0045,0.0080,False
5,rotation_mean_24h,0.0033,0.0082,False
6,pieces_produced_sum_24h,0.0043,0.0001,False
7,incident_count_prev_24h,0.0003,0.1723,False


## 5.2 Les 5 feux verts

Réentraîner a un coût et un risque. Le flow ne lance l'arbitrage que si **les 5 conditions**
sont réunies. On les évalue ici avec les mêmes fonctions que le flow, en lecture seule :

In [21]:
# 🟢 Lecture seule
from indusense.flows.retrain_flow import (
    _load_ingestion_batches,
    _load_reviews,
    check_confirmed_failures_quota,
    check_data_quality,
    check_drift_signal,
    check_new_terrain_batch,
    check_validation_quota,
)

reviews = _load_reviews(get_predictions_engine())
feux = {
    "1. ≥ 50 validations humaines": check_validation_quota(reviews),
    "2. ≥ 10 pannes confirmées": check_confirmed_failures_quota(reviews),
    "3. capteurs dans les bornes": check_data_quality(gold, CUTOFF),
    "4. dérive PSI > 0,25": check_drift_signal(gold, CUTOFF),
    "5. nouveau lot Gold": check_new_terrain_batch(_load_ingestion_batches(engine), CUTOFF),
}
pd.DataFrame(
    [(nom, "🟢 vert" if ok else "🔴 rouge", detail) for nom, (ok, detail) in feux.items()],
    columns=["feu", "état", "détail"],
)

,feu,état,détail
0,1. ≥ 50 validations humaines,🟢 vert,982 validations humaines (seuil 50)
1,2. ≥ 10 pannes confirmées,🟢 vert,727 pannes confirmées/déclarées (seuil 10)
2,3. capteurs dans les bornes,🟢 vert,aucune valeur hors bornes
3,"4. dérive PSI > 0,25",🟢 vert,PSI max temp_mean_24h=0.3841 (seuil 0.25)
4,5. nouveau lot Gold,🟢 vert,3 lot(s) Gold ingéré(s) depuis 2026-06-01


> **Le feu n°5 et la rupture n°1** : ce feu cherche un lot `gold` terminé dans le registre
> `ingestion_batch`. Avant, seul le notebook TP6 en écrivait. Désormais, le flow ETL (1.5)
> l'écrit lui-même : ce feu peut passer au vert **sans notebook**.

## 5.3 L'arbitrage : champion contre challenger

`arbitration.run_arbitration` entraîne un **challenger** sur des données enrichies des retours
techniciens, puis le compare au **champion** (le modèle servi) **cas par cas**, sur une période
qu'aucun des deux n'a vue :

- **gain** : le challenger a raison là où le champion se trompait ;
- **régression** : l'inverse, le plus grave ;
- **angle mort** : les deux se trompent.

Règle d'or : on ne remplace jamais un modèle sur un score global seul. Un challenger qui
régresse sans compensation massive est **rejeté**. Le calcul prend environ une minute ; rien
n'est écrit.

In [22]:
# 🟢 Lecture seule (entraîne un challenger en mémoire, ~1 min)
from indusense.arbitration import run_arbitration

arbitrage = run_arbitration(CUTOFF)
pd.Series(
    {k: arbitrage[k] for k in ["n_arbitrage", "gains", "regressions", "angles_morts",
                               "rappel_champion", "rappel_challenger", "decision"]},
    name="résultat",
).to_frame()

,résultat
n_arbitrage,2854
gains,10
regressions,6
angles_morts,39
rappel_champion,0.8814
rappel_challenger,0.9153
decision,REJET_DU_MODELE_N


## 5.4 Le mode fantôme et la bascule (non exécutés)

Si l'arbitrage accepte le challenger, `shadow.run_shadow_cycle` le fait tourner **en silence**
à côté du champion, sans alerter personne. Si l'observation confirme l'arbitrage,
`promote_challenger` remplace `model.joblib`, **après en avoir fait une sauvegarde** : revenir
en arrière est une simple copie.

On ne l'exécute pas ici : la bascule **remplacerait le modèle servi**.

> **Rupture n°5 (encore ouverte)** : le flow de réentraînement s'arrête à l'arbitrage ; le mode
> fantôme se lance à part (`scripts/run_shadow_mode.py`). Et une bascule ne met à jour ni
> l'étiquette DVC, ni l'image Docker publiée. Voir `cadrage_ruptures_package.md`.

---
# Bilan

| Étape | Ce qu'on a vérifié dans ce notebook |
|---|---|
| 1. Données | Le Gold reconstruit par le flow a le même contenu que celui versionné par DVC (seul l'identifiant de lot change) |
| 2. Modèle | Le modèle servi retrouve exactement ses métriques ; un réentraînement sur un autre PC en diffère légèrement |
| 3. Prédiction | Le facteur et le guichet donnent le même chiffre ; déposer deux fois ne crée aucun doublon |
| 4. Retour humain | Les verdicts techniciens recoupent les métriques ; un verdict n'est jamais écrasé |
| 5. Cycle de vie | Dérive, 5 feux verts et arbitrage se calculent avec les fonctions du flow |

### Où en sont les ruptures ?

| Rupture | État |
|---|---|
| n°1 : chaîne Bronze → Silver → Gold | **corrigée** : flow `indusense-etl`, commande `indusense etl` |
| n°3 : API isolée | **en partie** : logique de colonnes partagée (`b9e271e`) ; stockage et contrat à décider |
| n°5 : bascule détachée | **ouverte** |

### Pour aller plus loin

- Lancer le flow ETL (`RUN_ETL = True`), puis rejouer le notebook : l'empreinte reste identique.
- Lancer le facteur pour de vrai : `uv run --frozen python flows/pipeline.py`.
- Ouvrir l'interface technicien : `uv run --frozen streamlit run scripts/streamlit_review.py`.